## 函数集合
批量处理word文档中表格样式的一些函数集合

In [2]:
# ===================== 模块引用 =====================
import os
import re
from docx import Document
from docx.shared import Pt, RGBColor
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

print("模块加载完成")

模块加载完成


In [3]:
# ===================== 函数1：按首行/首列匹配提取目标表格 =====================
def get_target_tables(doc_obj, filter_feature):
    f_type, target_list = filter_feature
    matched = []
    for table in doc_obj.tables:
        if len(table.rows) == 0:
            continue
        if f_type == "row":
            row_text = [cell.text.strip() for cell in table.rows[0].cells]
        elif f_type == "col":
            row_text = [row.cells[0].text.strip() for row in table.rows]
        else:
            raise ValueError("filter_feature 第一个元素必须是 'row' 或 'col'")
        if row_text == target_list:
            matched.append(table)
    print(f"共提取符合要求的表格：{len(matched)}")
    return matched

In [ ]:
# ===================== 函数2：设置表格指定行/列底纹 =====================
def set_table_cell_shade(target_tables, row_index_list=None,
                         col_index_list=None, rgb_fill=(217, 225, 242), clear_shade=False):  #clear_shade=True时，清空选中单元格的底纹设置
    def _shade_cell(cell, rgb, clear):
        tc_pr = cell._tc.get_or_add_tcPr()
        old = tc_pr.find(qn("w:shd"))
        if clear:
            # 清除底纹：删除shd标签，无填充
            if old is not None:
                tc_pr.remove(old)
            return
        # 设置RGB底色
        hex_color = f"{rgb[0]:02X}{rgb[1]:02X}{rgb[2]:02X}"
        if old is not None:
            tc_pr.remove(old)
        shd = OxmlElement("w:shd")
        shd.set(qn("w:fill"), hex_color)
        tc_pr.append(shd)

    modify_run_count = 0
    if row_index_list is None and col_index_list is None:
        raise ValueError("row_index_list 与 col_index_list 至少传一个")
    for table in target_tables:
        for i, row in enumerate(table.rows):
            for j, cell in enumerate(row.cells):
                hit = False
                if row_index_list is not None and i in row_index_list:
                    _shade_cell(cell, rgb_fill, clear_shade)
                    hit = True
                if col_index_list is not None and j in col_index_list:
                    _shade_cell(cell, rgb_fill, clear_shade)
                    hit = True
                if hit:
                    modify_run_count += 1
    print(f"本次匹配并修改的单元格数量：{modify_run_count}")


In [1]:
#--------------函数：修改边框——————————————
def set_cell_border(cell, **kwargs):
    """
    设置Word单个单元格边框
    :param cell: table.cell 对象
    :param kwargs: top/bottom/left/right，每个参数是dict
        val: single(实线), dashed(虚线), none(无边框)
        sz: 粗细，单位 1/8磅 → sz=4 =0.5磅；sz=8=1磅
        color: 十六进制颜色，不带#，如 "000000"黑色
    """
    modify_run_count = 0  
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    tcBorders = OxmlElement('w:tcBorders')
    tcPr.append(tcBorders)

    for side in ('top', 'left', 'bottom', 'right'):
        if side in kwargs:
            data = kwargs[side]
            el = OxmlElement(f'w:{side}')
            el.set(qn('w:val'), data.get('val', 'single'))
            el.set(qn('w:sz'), str(data.get('sz', 4)))
            el.set(qn('w:color'), data.get('color', '000000'))
            el.set(qn('w:space'), str(data.get('space', 0)))
            tcBorders.append(el)
            modify_run_count+=1
    return modify_run_count


In [5]:
# ===================== 函数3：指定列对齐 =====================
from docx.enum.text import WD_ALIGN_PARAGRAPH

def set_table_column_alignment(tables, col_index_list, align_name:str):
    """
    对表格列表中指定列，批量设置段落对齐（支持字符串别名）
    :param tables: target_table_list
    :param col_index_list: 列索引列表，0开始，如 [1,3]
    :param align_name: 字符串：left / center / right / justify / distribute
    """
    align_map = {
        "left": WD_ALIGN_PARAGRAPH.LEFT,
        "center": WD_ALIGN_PARAGRAPH.CENTER,
        "right": WD_ALIGN_PARAGRAPH.RIGHT,
        "justify": WD_ALIGN_PARAGRAPH.JUSTIFY,
        "distribute": WD_ALIGN_PARAGRAPH.DISTRIBUTE
    }
    if align_name.lower() not in align_map:
        raise ValueError(f"对齐方式仅支持：{list(align_map.keys())}")
    align = align_map[align_name.lower()]

    modify_run_count = 0 
    for table in tables:
        for row in table.rows:
            for c_idx in col_index_list:
                if 0 <= c_idx < len(row.cells):
                    cell = row.cells[c_idx]
                    for para in cell.paragraphs:
                        para.alignment = align
                        modify_run_count += 1 
    print(f"共对齐段落数量：{modify_run_count}")


In [27]:
# ===================== 函数4：对目标表格文字做正则匹配并改样式 =====================
def regex_modify_text_style(target_tables, pattern, font_cn="宋体", font_en="Times New Roman",
                            font_size=9, font_rgb=(0, 0, 0), bold=False):
    modify_run_count = 0  
    reg = re.compile(pattern)
    def _set_run_font(run, f_cn, f_en, f_size, f_rgb, f_bold):
        rpr = run._element.get_or_add_rPr()
        rFonts = rpr.get_or_add_rFonts()
        rFonts.set(qn("w:eastAsia"), f_cn)
        rFonts.set(qn("w:ascii"), f_en)
        run.font.size = Pt(f_size)
        run.font.color.rgb = RGBColor(*f_rgb)
        run.font.bold = f_bold

    for table in target_tables:
        for row in table.rows:
            for cell in row.cells:
                for para in cell.paragraphs:
                    for run in para.runs:
                        if reg.search(run.text) is not None:
                            _set_run_font(run, font_cn, font_en, font_size, font_rgb, bold)
                            modify_run_count += 1 
    print(f"本次匹配并修改字体的run数量：{modify_run_count}")

In [6]:
# ===================== 函数5：另存为新文档 =====================
def save_new_docx(doc_obj, source_path, suffix, output_dir=None):
    dir_name, full_filename = os.path.split(source_path)
    name_no_ext, ext = os.path.splitext(full_filename)
    new_filename = f"{name_no_ext}{suffix}{ext}"

    if output_dir is None:
        output_path = os.path.join(dir_name, new_filename)
    else:
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, new_filename)

    doc_obj.save(output_path)
    print(f"已保存至：{output_path}")
    return output_path

## 调用示例
依次运行下面各 cell：输入文档及特征 → 提取表格 → 调用各函数 → 保存。

In [42]:
# 调用示例：加载word文档
source_doc_path = r"D:\文档资料\26年验收材料准备\需要自动处理的\治理提升工作报告.docx"                # 源文档完整路径
doc = Document(source_doc_path)
print("源文档加载完成")

源文档加载完成


In [ ]:
# 调用示例：定义提取数据表的特征
table_filter_feature_xuqiu = ("col", ["需求名称","需求描述", "需求背景", "基本信息","分类信息","处置说明","状态信息"])  # 输入特征，首行/首列，列的话输入col
table_filter_feature_wenti = ("col", ["记录名称","问题描述", "影响范围", "基本信息","问题分析","处置说明","状态信息"])  # 输入特征，首行/首列，列的话输入col

In [43]:
# 调用示例：提取目标表格
target_table_list_all=doc.tables
target_table_list_xuqiu = get_target_tables(doc, table_filter_feature_xuqiu)
target_table_list_wenti = get_target_tables(doc, table_filter_feature_wenti)

共提取符合要求的表格：0
共提取符合要求的表格：0


In [ ]:
# 调用示例③：设置目标行/列的底纹色
set_table_cell_shade(
    target_tables=target_table_list_all,
    row_index_list=[0],
    #col_index_list=[0],
    rgb_fill=(217, 217, 217),
    clear_shade=False         #如果选Ture,回清除底纹，传入颜色无效
)

本次匹配并修改的单元格数量：228
本次匹配并修改的单元格数量：51


In [47]:
# 调用示例: 给所有选中表设置边框
cell_count_sum = 0  # 记录修改的单元格数量
for table in target_table_list_all:
    for row in table.rows:
        for cell in row.cells:
            cell_count = set_cell_border(
                cell,
                top={"val":"single", "sz":0.25, "color":"B0B0B0"},
                bottom={"val":"single", "sz":0.25, "color":"B0B0B0"},
                left={"val":"single", "sz":0.25, "color":"B0B0B0"},
                right={"val":"single", "sz":0.25, "color":"B0B0B0"},
            )
            cell_count_sum += cell_count
print(f'已修改单元格{cell_count_sum}')


已修改单元格1696


In [48]:
# 调用示例：第1列居中
set_table_column_alignment(target_table_list_all, [0], "center")



共对齐段落数量：154


In [49]:
# 调用示例：对目标表格文字做正则匹配并修改文字样式
regex_modify_text_style(
    target_tables=target_table_list_all,
    pattern=r".*",
    font_cn="宋体",
    font_en="Times New Roman",
    font_size=9,
    font_rgb=(0, 0, 0),
    bold=False,
)


本次匹配并修改字体的run数量：449


In [50]:
# 调用示例⑤：保存到输出文件夹
output_suffix = "_processed"                     # 输出文件后缀
out_file = save_new_docx(
    doc_obj=doc,
    source_path=source_doc_path,  #不需要改
    suffix=output_suffix,
    output_dir=None,   #选择保存文件夹，不写的话存放在原文件夹
)

已保存至：D:\文档资料\26年验收材料准备\需要自动处理的\治理提升工作报告_processed.docx
